# Merge all sources + single embedding pass

Combines the two topic-tagged sources into **one** unified chunk set and runs
**one** embedding pass over it (the recommended approach — embed once, over
the union, not separately per source):

1. `input_data_topics/` — the 12 STEK documents, already chunked (978 chunks).
2. `scraped_data_topics/` — the website scrape: 117 HTML pages + 10 PDFs,
   **not yet chunked** (one file per page/doc).

Steps:
- Load input chunks as-is (reuse their topic tags).
- Strip headers from the scraped files, chunk them into the same ~250-word
  windows, and re-tag each chunk against the same 5 LDA topics.
- **Deduplicate**: skip scraped PDFs that are just re-hosted copies of the 12
  input documents, so the same text isn't embedded twice.
- Write one unified `vector_store/chunks.jsonl`.
- Embed every chunk once with a multilingual model and save `embeddings.npy`.
- Demo: cosine-similarity retrieval with an optional topic filter.

Embedding backend: **local `sentence-transformers`** with
`intfloat/multilingual-e5-base` (strong on German). Runs offline after a
one-time model download; no API key, no per-query cost.


## Config

In [3]:
import re
import json
import unicodedata
from pathlib import Path
from collections import Counter

INPUT_MANIFEST = Path("input_data_topics/manifest.json")
SCRAPED_MANIFEST = Path("scraped_data_topics/manifest.json")
LDA_TOPICS_PATH = Path("STEK2035-chatbot/pdfs/stek_output/texts/stek_results/lda_topics_k5.txt")

OUT_DIR = Path("vector_store")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_PATH = OUT_DIR / "chunks.jsonl"
EMB_PATH = OUT_DIR / "embeddings.npy"
META_PATH = OUT_DIR / "meta.json"

# same window size used to build input_data_topics (and the LDA training unit)
CHUNK_TARGET_TOKENS = 250
CHUNK_MIN_TOKENS = 80

EMB_MODEL_NAME = "intfloat/multilingual-e5-base"


## Step 1 — shared helpers

The same LDA-topic parser, cleaner, chunker, and tagger used to build
`input_data_topics`, so scraped chunks are processed identically.


In [4]:
def parse_lda_topics(path: Path) -> dict[int, list[str]]:
    topics = {}
    pattern = re.compile(r"Topic\s+(\d+):\s*(.+)")
    for line in path.read_text(encoding="utf-8").splitlines():
        m = pattern.match(line.strip())
        if m:
            topics[int(m.group(1))] = [w.strip() for w in m.group(2).split(",") if w.strip()]
    return topics


TOPIC_KEYWORDS = parse_lda_topics(LDA_TOPICS_PATH)


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = re.sub(r"([a-zäöüß])-\s*\n\s*([a-zäöüß])", r"\1\2", text)  # de-hyphenate
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def chunk_text(text: str, target_tokens: int, min_tokens: int) -> list[str]:
    words = text.split()
    chunks = [" ".join(words[i:i + target_tokens]) for i in range(0, len(words), target_tokens)]
    if len(chunks) >= 2 and len(chunks[-1].split()) < min_tokens:
        chunks[-2] = chunks[-2] + " " + chunks[-1]
        chunks.pop()
    return chunks


def tag_topics(text: str, topic_keywords: dict[int, list[str]]):
    lower = text.lower()
    matched_topics, matched_keywords, seen = [], [], set()
    for topic_id, keywords in sorted(topic_keywords.items()):
        hit = False
        for kw in keywords:
            needle = kw.replace("_", " ").lower()
            if re.search(rf"\b{re.escape(needle)}\b", lower):
                hit = True
                if kw not in seen:
                    seen.add(kw)
                    matched_keywords.append(kw)
        if hit:
            matched_topics.append(topic_id)
    return matched_topics, matched_keywords, len(matched_keywords)


def strip_header(raw: str) -> str:
    """Both source formats put a metadata header, then a blank line, then body."""
    norm = raw.replace("\r\n", "\n").replace("\r", "\n")
    parts = norm.split("\n\n", 1)
    return parts[1] if len(parts) == 2 else norm


## Step 2 — load the already-chunked input documents

Reads each input chunk's body from its file and reuses the topic tags that
were computed when `input_data_topics` was built.


In [5]:
def norm_key(name: str) -> str:
    """Normalized signature for duplicate detection across the two sources."""
    name = re.sub(r"%[0-9a-fA-F]{2}", " ", name)          # url-decode remnants
    name = name.rsplit(".", 1)[0]                           # drop extension
    name = re.sub(r"(?i)^\d+_pdf_", "", name)              # drop "12_pdf_" prefixes
    name = re.sub(r"(?i)^\d+_PDF_", "", name)
    return re.sub(r"[^a-z0-9]", "", name.lower())


input_manifest = json.loads(INPUT_MANIFEST.read_text(encoding="utf-8"))

records = []
input_keys = set()
for e in input_manifest:
    body = clean_text(strip_header(Path(e["path"]).read_text(encoding="utf-8")))
    if not body:
        continue
    input_keys.add(norm_key(e["document_id"]))
    records.append({
        "chunk_uid": f"doc::{e['document_id']}::{e['chunk_id']}",
        "source": "document",
        "origin": e["source_file"],
        "chunk_id": e["chunk_id"],
        "text": body,
        "topics": e["topics"],
        "relevance_score": e["relevance_score"],
    })

print(f"Loaded {len(records)} chunks from {len(input_manifest)} input entries")
print(f"Input document keys (for dedup): {sorted(input_keys)}")


Loaded 978 chunks from 978 input entries
Input document keys (for dedup): ['2015stadtentwicklungsplan', '20230728stek2035zukunftsreise2035', 'akstek2035dokumentation1sitzung091123kein', 'akstek2035dokumentation2sitzung250124', 'mro2035konzeptberichtbroschre', 'mro2035konzeptkarten', 'stek2035arbeitstreffen170624dokumentation', 'stek2035dokumentationonlinebeteiligungundaufsuchendeformate2024', 'stek2035dokumentationwegezudenzielen111224', 'stek2035dokumentationzukunftgestalten250624', 'steknachhaltigkeitsbericht2025', 'stekstadtentwicklungskonzept2035a3']


## Step 3 — chunk + tag the scraped website files (with dedup)

Each scraped file is one whole page/PDF, so it is chunked into the same
~250-word windows and each chunk is re-tagged. A scraped **PDF** is skipped
when it is just a re-hosted copy of one of the 12 input documents — detected
by a normalized-name match, plus an explicit alias list for copies whose
filename differs (e.g. `STEP 2015` == `2015_Stadtentwicklungsplan`). Every
skip/keep is printed so the decisions are auditable.


In [6]:
scraped_manifest = json.loads(SCRAPED_MANIFEST.read_text(encoding="utf-8"))

# Scraped-PDF url-tails known to duplicate an input doc but whose filename
# won't normalize to the same key. Edit this list if you spot more.
DUPLICATE_ALIASES = {
    "12_pdf_Step_2015_mit_Lesezeichen_mit_Vorwort_E_Wuerzner_s.pdf": "2015_Stadtentwicklungsplan",
}

skipped, kept_html, kept_pdf = [], 0, 0
for e in scraped_manifest:
    tail = e["url"].split("/")[-1]
    is_pdf = e["type"] == "pdf"

    # dedup only applies to scraped PDFs (HTML pages are unique website content)
    if is_pdf and (norm_key(tail) in input_keys or tail in DUPLICATE_ALIASES):
        skipped.append(tail)
        continue

    body = clean_text(strip_header(Path(e["path"]).read_text(encoding="utf-8")))
    if not body:
        continue

    source = "website_pdf" if is_pdf else "website_html"
    for cid, chunk in enumerate(chunk_text(body, CHUNK_TARGET_TOKENS, CHUNK_MIN_TOKENS)):
        topics, _, score = tag_topics(chunk, TOPIC_KEYWORDS)
        records.append({
            "chunk_uid": f"{source}::{tail}::{cid}",
            "source": source,
            "origin": e["url"],
            "chunk_id": cid,
            "text": chunk,
            "topics": topics,
            "relevance_score": score,
        })
    if is_pdf:
        kept_pdf += 1
    else:
        kept_html += 1

print(f"Skipped {len(skipped)} duplicate scraped PDFs:")
for s in skipped:
    print(f"    - {s}")
print(f"Kept {kept_html} HTML pages + {kept_pdf} unique scraped PDFs")
print(f"\nTotal unified chunks: {len(records)}")


Skipped 3 duplicate scraped PDFs:
    - 12_pdf_STEK_Nachhaltigkeitsbericht_2025.pdf
    - 12_pdf_STEK_Stadtentwicklungskonzept%202035_A3.pdf
    - 12_pdf_Step_2015_mit_Lesezeichen_mit_Vorwort_E_Wuerzner_s.pdf
Kept 117 HTML pages + 7 unique scraped PDFs

Total unified chunks: 1307


## Step 4 — write the unified chunk set

In [7]:
with CHUNKS_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

by_source = Counter(r["source"] for r in records)
print(f"Wrote {len(records)} chunks -> {CHUNKS_PATH}")
print("By source:", dict(by_source))

topic_counts = Counter()
for r in records:
    topic_counts.update(r["topics"])
print("Chunks per topic:", {t: topic_counts.get(t, 0) for t in sorted(TOPIC_KEYWORDS)})


Wrote 1307 chunks -> vector_store\chunks.jsonl
By source: {'document': 978, 'website_html': 189, 'website_pdf': 140}
Chunks per topic: {0: 733, 1: 794, 2: 793, 3: 834, 4: 837}


## Step 5 — single embedding pass

One-time install (uncomment). The first run downloads the model (~1 GB);
afterwards it runs fully offline.

`intfloat/multilingual-e5-base` expects a `"passage: "` prefix on documents
and a `"query: "` prefix on searches — the helpers below apply these so it
can't be gotten wrong. Embeddings are L2-normalized, so cosine similarity is
a plain dot product.


In [8]:
# %pip install sentence-transformers

import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# --- bootstrap: make this cell runnable on its own -------------------------
# If Steps 1-4 were run this turn, `records` is already in memory. Otherwise
# (e.g. running only Step 5) load the merged chunks from disk.
if "EMB_MODEL_NAME" not in globals():
    EMB_MODEL_NAME = "intfloat/multilingual-e5-base"
if "OUT_DIR" not in globals():
    OUT_DIR = Path("vector_store")
    CHUNKS_PATH, EMB_PATH, META_PATH = OUT_DIR / "chunks.jsonl", OUT_DIR / "embeddings.npy", OUT_DIR / "meta.json"
if "records" not in globals():
    records = [json.loads(ln) for ln in CHUNKS_PATH.read_text(encoding="utf-8").splitlines()]
    print(f"Loaded {len(records)} chunks from {CHUNKS_PATH}")
# ---------------------------------------------------------------------------

model = SentenceTransformer(EMB_MODEL_NAME)


def embed_passages(texts, batch_size=32):
    inputs = [f"passage: {t}" for t in texts]
    return model.encode(inputs, batch_size=batch_size, normalize_embeddings=True,
                        show_progress_bar=True).astype("float32")


def embed_query(query: str):
    return model.encode(f"query: {query}", normalize_embeddings=True).astype("float32")


texts = [r["text"] for r in records]
embeddings = embed_passages(texts)
np.save(EMB_PATH, embeddings)

META_PATH.write_text(json.dumps({
    "model": EMB_MODEL_NAME,
    "dim": int(embeddings.shape[1]),
    "count": int(embeddings.shape[0]),
    "normalized": True,
    "chunks_file": str(CHUNKS_PATH),
}, indent=2), encoding="utf-8")

print(f"Embedded {embeddings.shape[0]} chunks -> {embeddings.shape} saved to {EMB_PATH}")


Batches: 100%|██████████| 41/41 [13:24<00:00, 19.63s/it]


Embedded 1307 chunks -> (1307, 768) saved to vector_store\embeddings.npy


## Step 6 — retrieval demo (cosine similarity, optional topic filter)

At ~1.5k chunks a full numpy dot product is instant, so no FAISS/Chroma is
needed. `topic_filter` restricts results to chunks tagged with a given LDA
topic (0–4) — this is where the topic modeling pays off at query time.


In [9]:
def search(query: str, k: int = 5, topic_filter: int | None = None):
    q = embed_query(query)
    scores = embeddings @ q  # cosine, since everything is normalized

    order = np.argsort(-scores)
    hits = []
    for idx in order:
        r = records[idx]
        if topic_filter is not None and topic_filter not in r["topics"]:
            continue
        hits.append((float(scores[idx]), r))
        if len(hits) >= k:
            break
    return hits


for score, r in search("Wie wird bezahlbarer Wohnraum in Heidelberg geschaffen?", k=5):
    preview = r["text"][:180].replace("\n", " ")
    print(f"[{score:.3f}] {r['source']:12} topics={r['topics']} | {r['origin'][:55]}")
    print(f"          {preview}...\n")


[0.882] document     topics=[0, 3, 4] | 12_PDF_STEK 2035_Dokumentation_Online-Beteiligung und a
          nachhaltige Konzepte wie das des Mietshäuser Syndikats sollten speziell gefördert werden und eine aktive Bodenpolitik im Sinne einer Rekommunalisierung von Land betrieben werden. B...

[0.881] document     topics=[0, 3, 4] | 12_PDF_STEK 2035_Dokumentation_Online-Beteiligung und a
          allem günstigen Wohnraum für gering Verdienende und Studierende. OB Daten des Stadtplanungsamtes, Stand Januar 2024: - Entwicklungsgebiete Wohnraum: Schaffung von neuem Wohnraum fü...

[0.873] document     topics=[1, 2, 3, 4] | 12_PDF_STEK 2035_Dokumentation_Online-Beteiligung und a
          Beruf und Familie, Aufstiegsmöglichkeiten, interessante Forschungsprojekte. OB Günstigen und attraktiven Wohnraum schaffen. OB BEZAHLBARER!!! Wohnraum auch für Menschen ohne WBS!!!...

[0.871] document     topics=[0, 1, 2, 3, 4] | 12_pdf_STEK 2035_Dokumentation_Zukunft gestalten_25_06_
          muss bezahlb

## Next step

`vector_store/` now holds everything the chatbot's retriever needs:
`chunks.jsonl` (text + metadata), `embeddings.npy` (aligned row-for-row with
the jsonl), and `meta.json`. Wire `search()` into the RAG prompt: retrieve
top-k chunks for the user's question, pass their text as context to the LLM,
and cite `origin`. Re-run this notebook whenever the source data changes.
